# Task A — High-Performance CodeBERT Binary Classifier

**Goal:** Detect human-written vs machine-generated code (binary, label 0/1).

**Architecture:** CodeBERT `[CLS]` (768-d) → Deep MLP `768 → 256 → 128 → 64 → 1` with GELU, LayerNorm, dropout 0.3, sigmoid + calibrated threshold.

**Key fixes over previous notebooks:**
4. ✅ `max_length=512` with gradient checkpointing
5. ✅ Full data cleaning pipeline (dedup, encoding, contradictory labels)
6. ✅ Focal loss + LLRD + cosine warmup + early stopping + threshold calibration

In [ ]:
!pip install --upgrade transformers==4.45.0 huggingface_hub -q

In [ ]:
import os
os.environ["WANDB_DISABLED"] = "true"

import re
import math
import hashlib
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from collections import Counter
from tqdm import tqdm
from datasets import Dataset
from transformers import (
    RobertaTokenizer,
    RobertaModel,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding,
    get_cosine_schedule_with_warmup,
)
from transformers.modeling_outputs import SequenceClassifierOutput
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    f1_score,
)
import warnings
warnings.filterwarnings("ignore")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
print("All imports ready.")

## 1 · Data Loading & Cleaning

Full cleaning pipeline:
- Drop missing / placeholder rows
- Flag & remove encoding-suspect rows
- Exact deduplication via SHA-1
- Remove contradictory-label duplicates

In [ ]:
# ============================================================================
#  Inline preprocessing (mirrors clean_and_flag_task_a.py)
# ============================================================================

PLACEHOLDERS = {"unk", "na", "n/a", "-", "?", "none", "null", "nan"}
ENCODING_RE  = re.compile(r"Ã.|â\u20ac™|â\u20acœ|â\u20ac|\ufffd|\?\?\?")


def normalize_ws(s: pd.Series) -> pd.Series:
    return s.fillna("").astype(str).str.replace(r"\s+", " ", regex=True).str.strip()


def sha1_series(s: pd.Series) -> pd.Series:
    return s.map(lambda x: hashlib.sha1(x.encode("utf-8", "ignore")).hexdigest())


def clean_task_a(df: pd.DataFrame, text_col: str = "code",
                 label_col: str = "label") -> pd.DataFrame:
    n0 = len(df)

    # Drop missing-like rows
    raw = df[text_col]
    is_null = raw.isna()
    is_empty = raw.fillna("").astype(str).eq("")
    is_ws = raw.fillna("").astype(str).str.fullmatch(r"\s*")
    is_placeholder = raw.fillna("").astype(str).str.strip().str.lower().isin(PLACEHOLDERS)
    missing_mask = is_null | is_empty | is_ws | is_placeholder
    df = df[~missing_mask].reset_index(drop=True)
    print(f"  Dropped {missing_mask.sum()} missing/placeholder rows")

    # Drop encoding-suspect rows
    text_norm = normalize_ws(df[text_col])
    enc_mask = text_norm.str.contains(ENCODING_RE)
    n_enc = enc_mask.sum()
    df = df[~enc_mask].reset_index(drop=True)
    print(f"  Dropped {n_enc} encoding-suspect rows")

    # Exact deduplication
    text_norm = normalize_ws(df[text_col])
    exact_hash = sha1_series(text_norm)
    dup_mask = exact_hash.duplicated(keep="first")
    n_dup = dup_mask.sum()
    df = df[~dup_mask].reset_index(drop=True)
    print(f"  Dropped {n_dup} exact duplicates")

    # Drop contradictory-label groups
    text_norm = normalize_ws(df[text_col])
    exact_hash = sha1_series(text_norm)
    grp = pd.DataFrame({"h": exact_hash, "y": df[label_col].astype(str)})
    bad_hashes = set(grp.groupby("h")["y"].nunique().pipe(lambda s: s[s > 1]).index)
    contra_mask = exact_hash.isin(bad_hashes)
    n_contra = contra_mask.sum()
    df = df[~contra_mask].reset_index(drop=True)
    print(f"  Dropped {n_contra} contradictory-label rows")

    print(f"  Cleaning: {n0} → {len(df)} rows")
    return df


print("Preprocessing functions defined.")

In [ ]:
# ============================================================================
#  Load data, clean, stratified sample
# ============================================================================

SAMPLE_SIZE = 40_000          # ← 100K (up from 2K-10K)
RANDOM_SEED = 42

TRAIN_PATH = (
    "/kaggle/input/datasets/daniilor/semeval-2026-task13/"
    "SemEval-2026-Task13/task_a/task_a_training_set_1.parquet"
)
VAL_PATH = (
    "/kaggle/input/datasets/daniilor/semeval-2026-task13/"
    "SemEval-2026-Task13/task_a/task_a_validation_set.parquet"
)
TEST_PATH = (
    "/kaggle/input/datasets/daniilor/semeval-2026-task13/"
    "SemEval-2026-Task13/task_a/task_a_test_set_sample.parquet"
)

raw_df = pd.read_parquet(TRAIN_PATH)
val_raw_df = pd.read_parquet(VAL_PATH)
test_raw_df = pd.read_parquet(TEST_PATH)

print(f"Raw train: {len(raw_df)} rows, columns: {raw_df.columns.tolist()}")
print(f"Raw val:   {len(val_raw_df)} rows")
print(f"Raw test:  {len(test_raw_df)} rows")

raw_df["label"] = raw_df["label"].astype(int)
val_raw_df["label"] = val_raw_df["label"].astype(int)
test_raw_df["label"] = test_raw_df["label"].astype(int)

# --- Clean all splits ---
print("\nCleaning train:")
df_clean = clean_task_a(raw_df)
print("\nCleaning val:")
val_clean = clean_task_a(val_raw_df)
print("\nCleaning test:")
test_clean = clean_task_a(test_raw_df)

# --- Stratified sample of training data ---
if SAMPLE_SIZE < len(df_clean):
    df_sampled = df_clean.groupby("label", group_keys=False).apply(
        lambda x: x.sample(
            n=max(1, int(SAMPLE_SIZE * len(x) / len(df_clean))),
            random_state=RANDOM_SEED,
        )
    ).reset_index(drop=True)
else:
    df_sampled = df_clean.copy()

# --- Subsample validation (save compute) ---
VAL_SAMPLE_SIZE = 3_000
if VAL_SAMPLE_SIZE < len(val_clean):
    val_sampled = val_clean.groupby("label", group_keys=False).apply(
        lambda x: x.sample(
            n=max(1, int(VAL_SAMPLE_SIZE * len(x) / len(val_clean))),
            random_state=RANDOM_SEED,
        )
    ).reset_index(drop=True)
else:
    val_sampled = val_clean.copy()

train_df = df_sampled
val_df = val_sampled
test_df = test_clean

print(f"\nFinal splits:")
print(f"  Train: {len(train_df)} rows")
print(train_df["label"].value_counts().sort_index())
print(f"  Val: {len(val_df)} rows")
print(val_df["label"].value_counts().sort_index())
print(f"  Test: {len(test_df)} rows")
print(test_df["label"].value_counts().sort_index())

## 2 · Focal Loss

In [ ]:
class FocalLoss(nn.Module):
    """Binary focal loss — down-weights easy examples to focus on hard ones."""
    def __init__(self, gamma: float = 2.0, alpha: float = 0.25):
        super().__init__()
        self.gamma = gamma
        self.alpha = alpha

    def forward(self, logits, targets):
        targets = targets.float().view(-1)
        logits = logits.view(-1)
        bce = F.binary_cross_entropy_with_logits(logits, targets, reduction="none")
        probs = torch.sigmoid(logits)
        pt = torch.where(targets == 1, probs, 1 - probs)
        alpha_t = torch.where(targets == 1, self.alpha, 1 - self.alpha)
        return (alpha_t * (1 - pt) ** self.gamma * bce).mean()


print("FocalLoss defined.")

## 3 · Model Architecture

`[CLS](768) → 256 → 128 → 64 → 1` with GELU, LayerNorm, dropout 0.3, sigmoid output.

In [ ]:
class DeepHeadCodeBERT(nn.Module):
    """
    CodeBERT [CLS] (768-d) → deep MLP: 768 → 256 → 128 → 64 → 1
    GELU activations, LayerNorm, dropout, sigmoid output.
    """

    def __init__(self, model_name="microsoft/codebert-base", num_labels=1, dropout=0.3):
        super().__init__()
        self.num_labels = num_labels
        self.transformer = RobertaModel.from_pretrained(model_name)
        self.config = self.transformer.config
        hidden_size = self.config.hidden_size  # 768

        self.head = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.LayerNorm(256),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(256, 128),
            nn.LayerNorm(128),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(128, 64),
            nn.LayerNorm(64),
            nn.GELU(),
            nn.Dropout(dropout),

            nn.Linear(64, num_labels),
        )

        # Xavier init for head
        for m in self.head:
            if isinstance(m, nn.Linear):
                nn.init.xavier_uniform_(m.weight)
                nn.init.zeros_(m.bias)

    def gradient_checkpointing_enable(self, **kwargs):
        self.transformer.gradient_checkpointing_enable(**kwargs)

    def gradient_checkpointing_disable(self):
        self.transformer.gradient_checkpointing_disable()

    def forward(self, input_ids=None, attention_mask=None, labels=None, **kwargs):
        out = self.transformer(input_ids=input_ids, attention_mask=attention_mask)
        cls_vec = out.last_hidden_state[:, 0, :]  # (B, 768)
        logits = self.head(cls_vec)

        loss = None
        if labels is not None:
            loss = F.binary_cross_entropy_with_logits(
                logits.view(-1), labels.float().view(-1)
            )

        return SequenceClassifierOutput(loss=loss, logits=logits)


print("DeepHeadCodeBERT defined.")
print("  Architecture: [CLS](768) → 256 → 128 → 64 → 1")


## 4 · Tokenisation & Dataset Preparation



In [ ]:
MODEL_NAME = "microsoft/codebert-base"
MAX_LENGTH = 512                # ← up from 256

tokenizer = RobertaTokenizer.from_pretrained(MODEL_NAME)


def tokenize_fn(examples):
    """Tokenize for HF Dataset.map(batched=True).
    The DataCollatorWithPadding will handle conversion to tensors.
    """
    return tokenizer(
        examples["code"],
        truncation=True,
        max_length=MAX_LENGTH,
        # NO padding here — DataCollatorWithPadding pads per-batch (more efficient)
        # NO return_tensors — .map() needs plain lists
    )


def make_hf_dataset(df: pd.DataFrame) -> Dataset:
    ds = Dataset.from_pandas(df[["code", "label"]].reset_index(drop=True))
    ds = ds.map(tokenize_fn, batched=True, remove_columns=["code"],
                desc="Tokenising")
    ds = ds.rename_column("label", "labels")
    return ds


train_dataset = make_hf_dataset(train_df)
val_dataset   = make_hf_dataset(val_df)
test_dataset  = make_hf_dataset(test_df)

print(f"HF datasets ready — Train: {len(train_dataset)}, "
      f"Val: {len(val_dataset)}, Test: {len(test_dataset)}")
print(f"Sample keys: {list(train_dataset[0].keys())}")
print(f"Sample input_ids length: {len(train_dataset[0]['input_ids'])}")

## 5 · Training Setup

- Layer-wise learning rate decay (LLRD)
- Focal loss via custom Trainer
- Cosine schedule with warmup


In [ ]:
# ============================================================================
#  Layer-wise learning-rate decay (LLRD)
# ============================================================================

def get_layer_wise_lr_groups(
    model: DeepHeadCodeBERT,
    base_lr: float = 2e-5,
    head_lr: float = 1e-3,
    weight_decay: float = 0.01,
    llrd_factor: float = 0.95,
):
    opt_params = []
    no_decay = {"bias", "LayerNorm.weight", "LayerNorm.bias"}
    num_layers = model.config.num_hidden_layers  # 12

    # Embeddings
    emb_wd, emb_nowd = [], []
    for n, p in model.transformer.embeddings.named_parameters():
        (emb_nowd if any(nd in n for nd in no_decay) else emb_wd).append(p)
    emb_lr = base_lr * (llrd_factor ** num_layers)
    if emb_wd:
        opt_params.append({"params": emb_wd, "lr": emb_lr, "weight_decay": weight_decay})
    if emb_nowd:
        opt_params.append({"params": emb_nowd, "lr": emb_lr, "weight_decay": 0.0})

    # Encoder layers
    for i in range(num_layers):
        layer = model.transformer.encoder.layer[i]
        layer_lr = base_lr * (llrd_factor ** (num_layers - i))
        wd_p, nowd_p = [], []
        for n, p in layer.named_parameters():
            (nowd_p if any(nd in n for nd in no_decay) else wd_p).append(p)
        if wd_p:
            opt_params.append({"params": wd_p, "lr": layer_lr, "weight_decay": weight_decay})
        if nowd_p:
            opt_params.append({"params": nowd_p, "lr": layer_lr, "weight_decay": 0.0})

    # Classification head
    head_wd, head_nowd = [], []
    for n, p in model.head.named_parameters():
        (head_nowd if any(nd in n for nd in no_decay) else head_wd).append(p)
    if head_wd:
        opt_params.append({"params": head_wd, "lr": head_lr, "weight_decay": weight_decay})
    if head_nowd:
        opt_params.append({"params": head_nowd, "lr": head_lr, "weight_decay": 0.0})

    return opt_params


print("LLRD helper defined.")

In [ ]:
# ============================================================================
#  Custom Trainer: focal loss + LLRD + cosine schedule
# ============================================================================

class DeepHeadTrainer(Trainer):
    def __init__(self, *args, llrd_factor=0.95, head_lr=1e-3, focal_gamma=2.0, **kwargs):
        self.llrd_factor = llrd_factor
        self.head_lr = head_lr
        self.focal_loss = FocalLoss(gamma=focal_gamma)
        super().__init__(*args, **kwargs)

    def create_optimizer_and_scheduler(self, num_training_steps):
        param_groups = get_layer_wise_lr_groups(
            self.model,
            base_lr=self.args.learning_rate,
            head_lr=self.head_lr,
            weight_decay=self.args.weight_decay,
            llrd_factor=self.llrd_factor,
        )
        self.optimizer = torch.optim.AdamW(param_groups)

        warmup_steps = int(num_training_steps * self.args.warmup_ratio)
        self.lr_scheduler = get_cosine_schedule_with_warmup(
            self.optimizer,
            num_warmup_steps=warmup_steps,
            num_training_steps=num_training_steps,
        )

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        logits = outputs.logits
        loss = self.focal_loss(logits, labels)
        return (loss, outputs) if return_outputs else loss


print("DeepHeadTrainer defined.")

In [ ]:
# ============================================================================
#  Initialise model, configure training
# ============================================================================

DROPOUT      = 0.3
NUM_EPOCHS   = 5
BATCH_SIZE   = 16
BASE_LR      = 2e-5
HEAD_LR      = 1e-3
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
GRAD_CLIP    = 1.0
LLRD_FACTOR  = 0.95
OUTPUT_DIR   = "/kaggle/working/codebert_high_perf"

# --- Model ---
model = DeepHeadCodeBERT(
    model_name=MODEL_NAME,
    num_labels=1,
    dropout=DROPOUT,
).to(DEVICE)

total_p = sum(p.numel() for p in model.parameters())
train_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Model params: {total_p:,} total, {train_p:,} trainable")


def compute_metrics(eval_pred):
    preds, labels = eval_pred
    logits = np.squeeze(preds)
    probs = 1 / (1 + np.exp(-logits))        # sigmoid
    preds_binary = (probs >= 0.5).astype(int)

    acc = accuracy_score(labels, preds_binary)
    prec, rec, f1_weighted, _ = precision_recall_fscore_support(
        labels, preds_binary, average="weighted"
    )
    f1_macro = f1_score(labels, preds_binary, average="macro")

    return {
        "accuracy": acc,
        "f1_weighted": f1_weighted,
        "precision": prec,
        "recall": rec,
    }


# --- Training args ---
training_args = TrainingArguments(
    output_dir=OUTPUT_DIR,
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    weight_decay=WEIGHT_DECAY,
    logging_dir=f"{OUTPUT_DIR}/logs",
    logging_steps=100,
    eval_strategy="steps",
    eval_steps=500,
    save_strategy="steps",
    save_steps=500,
    load_best_model_at_end=True,
    greater_is_better=True,
    remove_unused_columns=False,
    learning_rate=BASE_LR,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    max_grad_norm=GRAD_CLIP,
    save_total_limit=2,
    report_to=[],
    fp16=True,
    gradient_checkpointing=True,       # ← saves VRAM for 512 tokens
    dataloader_num_workers=2,
)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = DeepHeadTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)],
    llrd_factor=LLRD_FACTOR,
    head_lr=HEAD_LR,
    focal_gamma=2.0,
)

print("\nTraining configuration:")
print(f"  Epochs: {NUM_EPOCHS}  Batch: {BATCH_SIZE}  Max length: {MAX_LENGTH}")
print(f"  Base LR: {BASE_LR}  Head LR: {HEAD_LR}  LLRD: {LLRD_FACTOR}")
print(f"  Gradient checkpointing: ON  FP16: ON")
print(f"  Training samples: {len(train_dataset):,}")

In [ ]:
# ============================================================================
#  Train!
# ============================================================================

print("Starting training ...")
trainer.train()

trainer.save_model()
tokenizer.save_pretrained(OUTPUT_DIR)
print(f"\nTraining complete. Best model saved to {OUTPUT_DIR}")

## 6 · Evaluation with Threshold Calibration

Calibrate sigmoid threshold on validation set, then evaluate on test.

In [ ]:
# ============================================================================
#  Evaluation utilities
# ============================================================================

SEEN_LANGS     = {"c++", "cpp", "python", "java"}
UNSEEN_LANGS   = {"go", "php", "c#", "csharp", "c", "javascript", "js"}
SEEN_DOMAINS   = {"algorithmic"}
UNSEEN_DOMAINS = {"research", "production"}


def _norm(v):
    return str(v).strip().lower()


def sigmoid_np(x):
    return 1 / (1 + np.exp(-x))


def calibrate_threshold(probs, labels):
    best_threshold = 0.5
    best_f1 = -1.0
    for threshold in np.arange(0.05, 0.96, 0.01):
        predictions = (probs >= threshold).astype(int)
        score = f1_score(labels, predictions, average="macro", zero_division=0)
        if score > best_f1:
            best_f1 = score
            best_threshold = float(threshold)
    return best_threshold, best_f1


def evaluate_by_category(df, tag="Model"):
    """Print classification metrics for 4 evaluation settings + overall."""
    lang_col = next(
        (c for c in df.columns if c.lower() in ("language", "lang", "programming_language")),
        None,
    )
    domain_col = next(
        (c for c in df.columns if c.lower() in ("domain", "task_type", "category")),
        None,
    )

    if "label" not in df.columns:
        print(f"[{tag}] No 'label' column — cannot evaluate.")
        return

    if lang_col is None or domain_col is None:
        print(f"[{tag}] Missing language/domain columns. Overall only:")
        print(classification_report(df["label"], df["prediction"]))
        return

    df = df.copy()
    df["_l"] = df[lang_col].apply(_norm)
    df["_d"] = df[domain_col].apply(_norm)

    settings = [
        ("(i)   Seen Lang & Seen Domain",    SEEN_LANGS,   SEEN_DOMAINS),
        ("(ii)  Unseen Lang & Seen Domain",   UNSEEN_LANGS, SEEN_DOMAINS),
        ("(iii) Seen Lang & Unseen Domain",   SEEN_LANGS,   UNSEEN_DOMAINS),
        ("(iv)  Unseen Lang & Unseen Domain", UNSEEN_LANGS, UNSEEN_DOMAINS),
    ]

    print(f"\n{'=' * 70}")
    print(f"  TEST RESULTS — {tag}")
    print(f"{'=' * 70}")

    for name, langs, domains in settings:
        mask = df["_l"].isin(langs) & df["_d"].isin(domains)
        sub = df[mask]
        n = len(sub)
        if n == 0:
            print(f"\n  {name}:  ** no samples **")
            continue
        y_t, y_p = sub["label"].values, sub["prediction"].values
        acc = accuracy_score(y_t, y_p)
        p, r, f1w, _ = precision_recall_fscore_support(y_t, y_p, average="weighted", zero_division=0)
        f1m = f1_score(y_t, y_p, average="macro", zero_division=0)
        print(f"\n  {name}  (n={n})")
        print(classification_report(y_t, y_p, zero_division=0))

    # Overall
    acc = accuracy_score(df["label"], df["prediction"])
    f1m = f1_score(df["label"], df["prediction"], average="macro", zero_division=0)
    f1w = f1_score(df["label"], df["prediction"], average="weighted", zero_division=0)
    print("=" * 70)


print("Evaluation utilities defined.")

In [ ]:
# ============================================================================
#  Inference helper
# ============================================================================

@torch.no_grad()
def predict_on_dataset(model, tokenizer, df, max_length=512, batch_size=32,
                       device="cuda", threshold=0.5):
    model.to(device)
    model.eval()
    codes = df["code"].tolist()
    probabilities = []
    predictions = []

    for i in tqdm(range(0, len(codes), batch_size), desc="Predicting"):
        batch = codes[i : i + batch_size]
        enc = tokenizer(
            batch, truncation=True, padding=True,
            max_length=max_length, return_tensors="pt",
        )
        fwd_kwargs = {
            "input_ids": enc["input_ids"].to(device),
            "attention_mask": enc["attention_mask"].to(device),
        }
        logits = model(**fwd_kwargs).logits.view(-1)
        batch_probs = torch.sigmoid(logits).cpu().numpy()
        probabilities.extend(batch_probs.tolist())
        predictions.extend((batch_probs >= threshold).astype(int).tolist())

    result = df.copy()
    result["probability"] = probabilities
    result["prediction"] = predictions
    return result


print("predict_on_dataset() defined.")

In [ ]:
# ============================================================================
#  Calibrate threshold on validation, then evaluate test
# ============================================================================

# --- Calibrate on validation ---
print(f"Calibrating threshold on validation set ({len(val_df)} samples) ...")
val_results = predict_on_dataset(
    model, tokenizer, val_df,
    max_length=MAX_LENGTH, batch_size=32, device=DEVICE, threshold=0.5,
)
val_threshold, val_best_f1 = calibrate_threshold(
    val_results["probability"].values,
    val_results["label"].values,
)

# --- Evaluate validation with tuned threshold ---
val_results["prediction"] = (val_results["probability"] >= val_threshold).astype(int)
evaluate_by_category(val_results, tag="CodeBERT-DeepHead (Validation, calibrated)")

# --- Evaluate test with tuned threshold ---
print(f"\nEvaluating on test set ({len(test_df)} samples) ...")
test_results = predict_on_dataset(
    model, tokenizer, test_df,
    max_length=MAX_LENGTH, batch_size=32, device=DEVICE, threshold=val_threshold,
)
evaluate_by_category(test_results, tag="CodeBERT-DeepHead (Test, calibrated)")

In [ ]:
# ============================================================================
#  Summary
# ============================================================================

print("\n" + "=" * 70)
print("  CONFIGURATION SUMMARY")
print("=" * 70)
print(f"  Model:             {MODEL_NAME}")
print(f"  Head:              768 → 256 → 128 → 64 → 1  (GELU + LayerNorm)")
print(f"  Training samples:  {len(train_dataset):,} (stratified from ~500K)")
print(f"  Val/Test:          official parquets (cleaned)")
print(f"  Max length:        {MAX_LENGTH}")
print(f"  Epochs:            {NUM_EPOCHS}")
print(f"  Batch size:        {BATCH_SIZE}")
print(f"  Base LR:           {BASE_LR}")
print(f"  Head LR:           {HEAD_LR}")
print(f"  LLRD factor:       {LLRD_FACTOR}")
print(f"  Weight decay:      {WEIGHT_DECAY}")
print(f"  Warmup ratio:      {WARMUP_RATIO}")
print(f"  Grad clip:         {GRAD_CLIP}")
print(f"  Dropout:           {DROPOUT}")
print(f"  Loss:              Focal (γ=2.0, α=0.25)")
print(f"  Precision:         fp16 (mixed)")
print(f"  Grad checkpointing: ON")
print(f"  Early stopping:    patience=3, metric=macro_f1")
print(f"  Scheduler:         cosine with warmup")
print(f"  Threshold:         {val_threshold:.2f} (calibrated on val)")
print("=" * 70)